# 1. Parameter design and training data

This is the first of three notebooks of a complete Bayesian study with `gpbayestools`
(see the [README](README.md) for an overview):

1. **Parameter design and training data** (this notebook)
2. [Emulator comparison](02_emulator_comparison.ipynb): training and validation of all emulators
3. [Closure tests](03_closure_tests.ipynb): Bayesian parameter estimation with pseudo-data

## The physics example

The example is the fit of deep-inelastic scattering (DIS) data from HERA in
[A. Andronic et al., JHEP 04 (2026) 185](https://doi.org/10.1007/JHEP04(2026)185)
([arXiv:2504.02726](https://arxiv.org/abs/2504.02726)).
The proton is described by the Golec-Biernat–Wüsthoff (GBW) saturation model, in which the
dipole amplitude depends on the saturation scale

$$Q_s^2(x) = Q_{s,0}^2\, x^{-\lambda}\, (1 - x).$$

The two model parameters are the saturation scale $Q_{s,0}$ and the exponent $\lambda$ of its
energy dependence. The observables are the reduced cross sections $\sigma_r$ at the 100 data
points of the combined H1 and ZEUS measurement with $2\ \mathrm{GeV}^2 \le Q^2 \le 22\ \mathrm{GeV}^2$
and $x \le 0.01$.

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from gpbayestools import parse_model_parameter_file

data_dir = Path("data")
figure_dir = Path("figures")
figure_dir.mkdir(exist_ok=True)

parameter_file = data_dir / "parameters_DIS.txt"
training_file = data_dir / "training_data_DIS.pkl"

## The parameter file

The parameter file defines the names, the plot labels and the prior ranges of the model
parameters, one line `name: label, min, max` per parameter. The labels must not contain commas.

In [ ]:
print(parameter_file.read_text())
pardict = parse_model_parameter_file(parameter_file)
pardict

## Generating the design

The model is run at design points that fill the parameter space. `gpbayestools.design.Design`
generates a maximum projection Latin-hypercube design (`method="maxpro"`, R package MaxPro) or a
maximin design (`method="maximin"`, R package lhs) and writes one input file per design point.
R with the package is needed, so the cell is not executed here; the training data below were
computed for 1000 design points generated in this way:

```python
from gpbayestools.design import Design

design = Design(parameter_file, n_points=1000, seed=1, method="maxpro")
design.write_files("design_input")  # writes design_input/main/parameter_000, ...
```

The design is cached in `cache/lhs/` (the directory is set by the environment variable
`WORKDIR`). A validation design needs `validation=True` and a different seed.
MaxPro designs are ordered with `MaxProRunOrder`, so that the first points of the design also fill
the parameter space. This is why the first 500 points can be used for the training and the
remaining 500 for the validation.

## The training data

The model output is stored in a pickle file with a dictionary
`{event_id: {"parameter": array (n_parameters,), "obs": array (2, n_obs)}}`. The first row of
`obs` holds the values of the observables, the second row their uncertainties. The emulators
sort the design points by their integer `event_id`.

In [ ]:
with open(training_file, "rb") as f:
    training_data = pickle.load(f)

ids = sorted(training_data, key=int)
design = np.array([training_data[i]["parameter"] for i in ids])
values = np.array([training_data[i]["obs"][0] for i in ids])
errors = np.array([training_data[i]["obs"][1] for i in ids])
print(f"{len(ids)} design points, {values.shape[1]} observables")
print("first entry:", ids[0], training_data[ids[0]]["parameter"], training_data[ids[0]]["obs"].shape)
print(f"values between {values.min():.3g} and {values.max():.3g}")
print(f"relative uncertainties between {np.min(errors / values):.4f} and {np.max(errors / values):.4f}")

The GBW model is evaluated without statistical fluctuations, so the values are smooth functions of
the parameters. The uncertainty of 6.59% of every value is the **model uncertainty** assigned in
the paper, not a statistical error of the simulation. This matters for the emulators that use the
uncertainties of the training data (PCSK treats them as noise, see notebook 2).

The observables span more than two orders of magnitude, which motivates the log transformation
compared in notebook 2.

In [ ]:
n_train = 500
labels = [val[0] for val in pardict.values()]

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(*design[:n_train].T, s=8, label=f"training ({n_train})")
ax.scatter(*design[n_train:].T, s=8, marker="x", label=f"validation ({len(design) - n_train})")
ax.set_xlabel(labels[0])
ax.set_ylabel(labels[1])
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2, frameon=False)
fig.tight_layout()
fig.savefig(figure_dir / "01_design.png", dpi=150)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = plt.cm.viridis((design[:, 0] - 0.1) / 0.7)
for i in range(0, len(design), 5):
    ax.plot(values[i], color=colors[i], lw=0.5, alpha=0.6)
ax.set_yscale("log")
ax.set_xlabel("observable index (HERA data point)")
ax.set_ylabel(r"$\sigma_r$")
sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(0.1, 0.8))
fig.colorbar(sm, ax=ax, label=labels[0])
ax.set_title("Model output at every 5th design point")
fig.tight_layout()
fig.savefig(figure_dir / "01_training_data.png", dpi=150)

The next notebook trains the emulators on the first 500 design points and validates them on the
remaining 500.